In [13]:
import torch
from torch import nn
import math

In [24]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, X):
        print("\n=== INPUT ===")
        print("X shape:", X.shape)
        print("X sample:", X[0, :2, :])

        batch_size, seq_len, _ = X.shape

        # ---- 1. Compute Q, K, V ----
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)

        print("\n=== AFTER LINEAR PROJECTIONS ===")
        print("Q shape:", Q.shape)
        print("K shape:", K.shape)
        print("V shape:", V.shape)

        # ---- 2. Split into multiple heads ----
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k)

        print("\n=== AFTER SPLIT INTO HEADS ===")
        print("Q shape:", Q.shape)  # (B, T, h, d_k)
        print("K shape:", K.shape)  
        print("V shape:", V.shape)  

        # transpose to (B, h, T, d_k)
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        print("\n=== AFTER TRANSPOSE ===")
        print("Q shape:", Q.shape)  # (B, h, T, d_k)
        print("K shape:", K.shape)  
        print("V shape:", V.shape)  

        # ---- 3. Compute attention scores ----
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        print("\n=== ATTENTION SCORES ===")
        print("scores shape:", scores.shape)  # (B, h, T, T)

        # ---- 4. Apply causal mask ----
        mask = torch.tril(torch.ones(seq_len, seq_len))
        print("\nMask:\n", mask)

        scores = scores.masked_fill(mask == 0, float('-inf'))

        print("\n=== MASKED SCORES ===")
        print("scores after mask:", scores)

        # ---- 5. Softmax → attention weights ----
        attn = torch.softmax(scores, dim=-1)

        print("\n=== ATTENTION WEIGHTS ===")
        print("attn shape:", attn.shape)
        print("attn sample:", attn[0, 0])

        # ---- 6. Weighted sum of values ----
        out = torch.matmul(attn, V)

        print("\n=== WEIGHTED VALUES ===")
        print("out shape:", out.shape)  # (B, h, T, d_k)

        # ---- 7. Concatenate heads ----
        out = out.transpose(1, 2)

        print("\n=== AFTER TRANSPOSE BACK ===")
        print("out shape:", out.shape)  # (B, T, h, d_k)

        out = out.contiguous().view(batch_size, seq_len, self.d_model)

        print("\n=== AFTER CONCAT HEADS ===")
        print("out shape:", out.shape)  # (B, T, d_model)

        # ---- 8. Final projection ----
        out = self.W_o(out)

        print("\n=== FINAL OUTPUT ===")
        print("out shape:", out.shape)
        print("out sample:", out[0, :2, :])

        return out


def test():
    torch.manual_seed(42)

    batch_size, seq_length, dim = 2, 5, 4
    X = torch.randn((batch_size, seq_length, dim))
    
    dim, head_num = 4, 2
    mh_attention = MultiHeadSelfAttention(dim, head_num)
    
    output = mh_attention(X)
    
    print("\n=== RETURNED OUTPUT ===")
    print(output)


test()


=== INPUT ===
X shape: torch.Size([2, 5, 4])
X sample: tensor([[ 1.9269,  1.4873,  0.9007, -2.1055],
        [ 0.6784, -1.2345, -0.0431, -1.6047]])

=== AFTER LINEAR PROJECTIONS ===
Q shape: torch.Size([2, 5, 4])
K shape: torch.Size([2, 5, 4])
V shape: torch.Size([2, 5, 4])

=== AFTER SPLIT INTO HEADS ===
Q shape: torch.Size([2, 5, 2, 2])
K shape: torch.Size([2, 5, 2, 2])
V shape: torch.Size([2, 5, 2, 2])

=== AFTER TRANSPOSE ===
Q shape: torch.Size([2, 2, 5, 2])
K shape: torch.Size([2, 2, 5, 2])
V shape: torch.Size([2, 2, 5, 2])

=== ATTENTION SCORES ===
scores shape: torch.Size([2, 2, 5, 5])

Mask:
 tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

=== MASKED SCORES ===
scores after mask: tensor([[[[-1.0098,    -inf,    -inf,    -inf,    -inf],
          [-0.6212, -0.5059,    -inf,    -inf,    -inf],
          [-0.3735, -0.9382,  0.0257,    -inf,    -inf],
          [ 0.1083, -0.192